# NB02D2 — V2-inspired photometric augmentation (no image correction)

**Objective:** A clean experiment testing whether V2's successful *training-time illumination robustness recipe* improves the frozen NB02B M2 baseline under the same 30-circle / three-spatial-fold protocol.

**Only change:** training-time independent brightness (0.70–1.30), contrast (0.80–1.20), gamma (0.75–1.35), small per-channel gain (0.92–1.08), plus a broad simulated multiplicative shadow (probability 0.55; shadow darkness 0.45–0.90). Adapted directly from V2 Notebook 04, Section 04. RGB mosaics and inference are **unaltered native RGB**. No site gain, no per-chip brightness normalization, no Gaussian image preprocessing. Model, loss, fold splits, original cached chips, optimizer, learning rate and 25 epochs follow NB02B M2. The historical V2 experiment evaluated crowns and cannot establish improvement on V3 fractional-cover evaluation; this is the direct V3 comparison.

**Fast path:** Load the already built NB02B chip caches without touching enormous mosaics during training. Train 3 new M2 checkpoints with resumable output, evaluate 30 circles, and compare matched-pixel results to frozen NB02B. Does **not** modify the original NB02D local-illumination experiment. Outputs: `Outputs/V3/02D2_v2_photometric_M2/`.


## 01 — Inputs and NB01 spatial folds


In [ ]:
%pip -q install rasterio geopandas pyogrio shapely pandas matplotlib scikit-learn segmentation-models-pytorch
from pathlib import Path
import json, warnings, math, random, time
import numpy as np, pandas as pd, geopandas as gpd, rasterio, matplotlib.pyplot as plt
import torch, torch.nn as nn, torch.nn.functional as F
from rasterio.windows import Window
from rasterio.features import rasterize
from shapely.geometry import box
from shapely.ops import unary_union
from shapely import make_valid
from torch.utils.data import DataLoader, Dataset
from sklearn.metrics import average_precision_score, roc_auc_score
from IPython.display import display
from google.colab import drive
if not Path('/content/drive/MyDrive').exists():drive.mount('/content/drive')
ROOT=next((p for p in [Path('/content/drive/MyDrive/Colab Notebooks/Musanga/Final'),
                         Path('/content/drive/MyDrive/Musanga/Final')] if p.is_dir()),None)
assert ROOT is not None
BASELINE=ROOT/'Outputs/V3/02B_spatial3fold_no_MOS34'
OUT=ROOT/'Outputs/V3/02D2_v2_photometric_M2';OUT.mkdir(parents=True,exist_ok=True)
REF=ROOT/'Outputs/V3/01_reference_data_no_MOS34'
A=json.loads((ROOT/'Outputs/V3/00_input_audit/00_input_manifest.json').read_text())
assert A.get('schema_version')=='v3_input_audit_5_nadine_circles'
assert A['exhaustive_annotations'].get('all_annotation_polygons_musanga') is True
assert int(A['exhaustive_annotations']['verified_exhaustive_circle_count'])==34
EXCLUDED_MOSAIC='Mos_3_4_transparent_mosaic_group1_georef.tif'
mosaics={m['file']:m for m in A['mosaics'] if m['file']!=EXCLUDED_MOSAIC}
splits=pd.read_csv(REF/'01_cv_window_manifest.csv')
links=pd.read_csv(REF/'01_circle_spatial_folds.csv')
repairs=pd.read_csv(REF/'01_geometry_repairs.csv')
assert links.circle_id.nunique()==30
assert links.groupby('site').circle_id.nunique().to_dict()=={'Baego':7,'MOS':11,'Yangambi':7,'Yoko':5}
assert links.circle_id.is_unique
assert len(mosaics)==5,[(k,v['site']) for k,v in mosaics.items()]
TEST_SITE='SPATIAL_CV'  # training includes all four sites
FOLD=0;SEED=20261016;CROP=256;EPOCHS=25;BATCH=8
MAX_PARTIAL_PER_MOSAIC=250
MAX_EXHAUSTIVE_PER_SITE=260
HOLDOUT_COUNTS={'Baego':1,'MOS':2,'Yangambi':2,'Yoko':2}
assert torch.cuda.is_available(),'Select Runtime > Change runtime type > GPU for the pilot.'
device='cuda';torch.manual_seed(SEED);np.random.seed(SEED);random.seed(SEED)
print('PRIMARY: 3-fold within-site CV, partial-only M1 vs added-exhaustive M2')
print('GPU',torch.cuda.get_device_name(0),'| fold',FOLD)
print('Train/val candidates:',splits[splits.cv_fold==FOLD].split.value_counts().to_dict())
print('Files:',ROOT,'| output:',OUT)




print('Mosaics:',[(name,m['site']) for name,m in mosaics.items()])


## 02 — Load annotations and ImageNet-pretrained ResNet18 U-Net

In [ ]:
def pieces(g):
    if g.geom_type in ('Polygon','MultiPolygon'):return [g]
    if hasattr(g,'geoms'):return [p for q in g.geoms for p in pieces(q)]
    return []
def polygons(path):
    src=gpd.read_file(path);inds=[];shapes=[]
    for i,g in enumerate(src.geometry):
        if g is None or g.is_empty:continue
        if not g.is_valid:
            try:
                fixed=unary_union(pieces(make_valid(g)))
                change=abs(fixed.area-g.area)/max(fixed.area,g.area,1e-12)
                if change>(.01 if Path(path) in (Path(A['exhaustive_annotations']['complete_roi_boundaries']),Path(A['exhaustive_annotations']['positive_polygons']),Path(A['roi'])) else .05) or not fixed.is_valid:continue
                g=fixed
            except Exception:continue
        inds.append(i);shapes.append(g)
    x=src.iloc[inds].copy().reset_index(drop=True)
    x.geometry=shapes
    x['original_row']=inds
    return x
roi=polygons(A['roi'])
circles=polygons(A['exhaustive_annotations']['complete_roi_boundaries'])
positive=polygons(A['exhaustive_annotations']['positive_polygons'])
assert len(circles)==34 and len(positive)>0, (
    f'Unexpected reference geometries: circles={len(circles)} (expected 34), '
    f'Musanga polygons={len(positive)}; do not train until reconciled')
circle_ids=set(links.circle_id.astype(int))
circles=circles[circles.original_row.isin(circle_ids)].copy()
tree={}
for site,path in A['shared_annotations'].items():
    t=polygons(path)
    cls=pd.to_numeric(t.Class_ID,errors='coerce')
    t['class01']=np.where(cls.eq(1),1,np.where(cls.notna(),0,-1))
    tree[site]=t
geo={}
for name,m in mosaics.items():
    with rasterio.open(m['path']) as src:
        c=circles.to_crs(src.crs).copy()
        c['fold']=c.original_row.map(links.set_index('circle_id').fold)
        c['owner']=c.original_row.map(links.set_index('circle_id').mosaic)
        geo[name]=dict(site=m['site'],path=m['path'],transform=src.transform,
            crs=src.crs,roi=roi.to_crs(src.crs),
            circles=c,positive=positive.to_crs(src.crs),trees=tree[m['site']].to_crs(src.crs))



In [ ]:
class ResNet18UNet(nn.Module):
    """Original main-branch U-Net: ImageNet-pretrained ResNet18 encoder, RGB in, 1 logit out."""
    def __init__(self):
        super().__init__()
        import segmentation_models_pytorch as smp
        self.net=smp.Unet(encoder_name='resnet18',encoder_weights='imagenet',
                          in_channels=3,classes=1)
    def forward(self,x):
        return self.net(x)[:,0]

class CropData(Dataset):
    def __init__(self,rows,augment=False):self.rows=rows;self.augment=augment
    def __len__(self):return len(self.rows)
    def __getitem__(self,i):
        d=self.rows[i];image=d['rgb'];y=d['y'];w=d['w']
        if self.augment:
            rng=np.random.default_rng(SEED+i+int(torch.randint(0,1000000,(1,)).item()))
            k=int(rng.integers(4));image=np.rot90(image,k);y=np.rot90(y,k);w=np.rot90(w,k)
            if rng.random()<.5:image=image[:,::-1];y=y[:,::-1];w=w[:,::-1]
        x=torch.tensor(np.ascontiguousarray(image.transpose(2,0,1)),dtype=torch.float32)
        return x,torch.tensor(np.ascontiguousarray(y)),torch.tensor(np.ascontiguousarray(w))
def masked_loss(logits,y,w):
    loss=F.binary_cross_entropy_with_logits(logits,y,reduction='none')
    return (loss*w).sum()/w.sum().clamp_min(1.)
print('ResNet18 U-Net ready (native 256×256 RGB crops).')




## 03 — Per-fold spatial exclusion (reconfigured for each training fold)

Use NB01 circle assignments, not site holdouts. Training crop footprint must not intersect any validation circle plus 15 m, across both MOS mosaics.


In [ ]:
assert links.groupby(['site','fold']).circle_id.nunique().ge(1).all()
assert sorted(links.fold.unique().astype(int).tolist())==[0,1,2]
HOLDOUT=set();DEV=set();BLOCKED={}
split_df=links.iloc[:0].copy()
print('NB01 circle folds:',links.groupby(['site','fold']).size().unstack(fill_value=0).to_dict())


## 04 — Resumable caches and direct RGB window reads

Source data, 256-pixel crops, augmentation, model architecture, optimizer and 25-epoch training remain unchanged. Cache fingerprint incorporates fold, supervision type and training-label selection. Direct Drive window reads avoid full-mosaic transfers.


In [ ]:
from rasterio.windows import Window
from torch.utils.data import Dataset,DataLoader
candidate=pd.read_csv(REF/'01_candidate_windows.csv')
candidate=candidate[candidate.mosaic.isin(mosaics)].copy()
all_circle_buffers={}
for name,d in geo.items():
    cs=circles.to_crs(d['crs'])
    all_circle_buffers[name]=[g.buffer(15.) for g in cs.geometry]
    d['dev_circles']=cs[cs.original_row.isin(DEV)]
    d['holdout_circles']=cs[cs.original_row.isin(HOLDOUT)]
    d['positive_local']=positive.to_crs(d['crs'])
    d['trees_local']=tree[d['site']].to_crs(d['crs'])
    d['tree_positive']=d['trees_local'].loc[d['trees_local'].class01==1]
    d['tree_negative']=d['trees_local'].loc[d['trees_local'].class01==0]
    d['roi_local']=roi.to_crs(d['crs'])

def burn_window(geometries,name,win):
    """Rasterize geometries intersecting a crop, using the spatial index.

    Accept GeoSeries OR GeoDataFrame; always iterate actual Shapely geometries,
    never GeoDataFrame column names.
    """
    d=geo[name]
    series=geometries.geometry if isinstance(geometries,gpd.GeoDataFrame) else geometries
    assert isinstance(series,gpd.GeoSeries),type(series)
    bb=box(*rasterio.windows.bounds(win,d['transform']))
    ids=series.sindex.query(bb,predicate='intersects')
    shape=(int(win.height),int(win.width))
    if len(ids)==0:return np.zeros(shape,dtype=bool)
    selected=series.iloc[ids]
    return rasterize(((g,1) for g in selected.values
                      if g is not None and not g.is_empty),
        out_shape=shape,transform=rasterio.windows.transform(win,d['transform']),
        fill=0,dtype='uint8').astype(bool)

_ACTIVE_RASTER=None
_ACTIVE_NAME=None

def rgb_for_window(name,win):
    if _ACTIVE_RASTER is not None and _ACTIVE_NAME==name:
        raw=_ACTIVE_RASTER.read([1,2,3],window=win,masked=True)
    else:
        with rasterio.open(geo[name]['path']) as ds:
            raw=ds.read([1,2,3],window=win,masked=True)
    valid=~np.any(np.ma.getmaskarray(raw),axis=0)
    a=raw.filled(0).astype('float32')
    valid &= np.any(a!=0,axis=0)&np.any(a!=255,axis=0)
    valid &= ~((a[0]==0)&(a[1]==0)&(a[2]==255))
    return np.clip(a/255.,0,1).transpose(1,2,0).astype('float32'),valid

def training_crop(rec,model):
    name=rec.mosaic;d=geo[name]
    if rec.height<CROP or rec.width<CROP:return None
    rr=int(rec.row)+(int(rec.height)-CROP)//2
    cc=int(rec.col)+(int(rec.width)-CROP)//2
    win=Window(cc,rr,CROP,CROP)
    bb=box(*rasterio.windows.bounds(win,d['transform']))
    if any(bb.intersects(g) for g in BLOCKED[name]):return None
    if model=='M1' and any(bb.intersects(g) for g in all_circle_buffers[name]):return None
    if model=='M2' and rec.kind=='exhaustive' and int(rec.circle_id) not in DEV:return None
    inside=burn_window(d['roi_local'].geometry,name,win)
    pos_tree=burn_window(d['tree_positive'],name,win)
    neg_tree=burn_window(d['tree_negative'],name,win)
    if model=='M1':
        full=np.zeros_like(inside);pos_ex=np.zeros_like(inside)
    else:
        full=burn_window(d['dev_circles'].geometry,name,win)&inside
        pos_ex=burn_window(d['positive_local'].geometry,name,win)&full
    conflict=(pos_tree&neg_tree)| (pos_ex&neg_tree) | (full&~pos_ex&pos_tree)
    target=(pos_ex|(~full&pos_tree)).astype('float32')
    weight=(full|((pos_tree|neg_tree)&~full))&inside&~conflict
    img,valid=rgb_for_window(name,win)
    weight &= valid
    if int(weight.sum())<250:return None
    return dict(image=img,target=target,weight=weight.astype('float32'),
                site=d['site'],kind=rec.kind,mosaic=name)

def _pack_chips(path,items,signature):
    """Atomic, compressed uint8 cache of already-prepared chips."""
    if not items:return
    tmp=path.with_name(path.name+'.tmp.npz')
    try:
        np.savez_compressed(tmp,key=np.array(signature),
            rgb=np.stack([np.rint(np.clip(r['image'],0,1)*255).astype('uint8') for r in items]),
            target=np.stack([r['target'].astype('uint8') for r in items]),
            weight=np.stack([r['weight'].astype('uint8') for r in items]),
            site=np.array([r['site'] for r in items],dtype='U12'),
            kind=np.array([r['kind'] for r in items],dtype='U16'))
        tmp.replace(path)
    finally:
        tmp.unlink(missing_ok=True)

def _unpack_chips(path,signature):
    with np.load(path,allow_pickle=False) as z:
        assert str(z['key'].item())==signature
        rgb=z['rgb'];target=z['target'];weight=z['weight']
        sites=z['site'].astype(str);kinds=z['kind'].astype(str)
        assert rgb.ndim==4 and rgb.shape[1:]==(CROP,CROP,3)
        assert target.shape==weight.shape==rgb.shape[:3]
        return [dict(image=rgb[i].astype('float32')/255.,
                     target=target[i].astype('float32'),
                     weight=weight[i].astype('float32'),
                     site=str(sites[i]),kind=str(kinds[i]))
                for i in range(len(rgb))]

import shutil
from contextlib import contextmanager
LOCAL_STAGE=Path('/content/musanga_02f_stage')
LOCAL_STAGE.mkdir(parents=True,exist_ok=True)
MIN_FREE_AFTER_STAGE=2*1024**3
USE_LOCAL_STAGE=True

@contextmanager
def staged_mosaic(name):
    """Stage one raster on SSD if it fits; use one open raster handle per mosaic."""
    global _ACTIVE_RASTER,_ACTIVE_NAME
    source=Path(geo[name]['path'])
    assert source.exists(),source
    free=shutil.disk_usage('/content').free
    needed=source.stat().st_size
    local=LOCAL_STAGE/(name.replace('/','_'))
    can_stage=USE_LOCAL_STAGE and free>=needed+MIN_FREE_AFTER_STAGE
    selected_path=source
    if can_stage:
        print(f'STAGE {name}: {needed/1e9:.2f} GB from Drive to local SSD '
              f'(available {free/1e9:.1f} GB)',flush=True)
        t0=time.perf_counter()
        tmp=local.with_name(local.name+'.part')
        try:
            with source.open('rb') as inp,tmp.open('wb') as out:
                shutil.copyfileobj(inp,out,length=16*1024*1024)
            assert tmp.stat().st_size==needed
            tmp.replace(local)
            selected_path=local
            print(f'STAGED {name} in {time.perf_counter()-t0:.1f}s',flush=True)
        except Exception as exc:
            tmp.unlink(missing_ok=True)
            local.unlink(missing_ok=True)
            print(f'Local copy failed; using original Drive image: {exc}',flush=True)
    else:
        print(f'STAGE SKIPPED {name}: image {needed/1e9:.2f} GB, '
              f'free SSD {free/1e9:.2f} GB; using one persistent Drive handle',
              flush=True)
    assert _ACTIVE_RASTER is None,'Another mosaic is already open.'
    try:
        with rasterio.open(selected_path) as ds:
            assert ds.count>=3 and ds.transform==geo[name]['transform']
            assert ds.crs==geo[name]['crs']
            _ACTIVE_RASTER=ds
            _ACTIVE_NAME=name
            yield
    finally:
        _ACTIVE_RASTER=None
        _ACTIVE_NAME=None
        if selected_path==local:
            local.unlink(missing_ok=True)
            print(f'RELEASED local SSD for {name}',flush=True)

class CropDataset(Dataset):
    def __init__(self,rows):self.rows=rows
    def __len__(self):return len(self.rows)
    def __getitem__(self,i):
        q=self.rows[i];x=q['image'].copy();y=q['target'].copy();w=q['weight'].copy()
        k=int(torch.randint(0,4,(1,)).item())
        x=np.rot90(x,k);y=np.rot90(y,k);w=np.rot90(w,k)
        if torch.rand(1).item()<.5:x=x[:,::-1];y=y[:,::-1];w=w[:,::-1]
        return (torch.from_numpy(np.ascontiguousarray(x.transpose(2,0,1))),
                torch.from_numpy(np.ascontiguousarray(y)),torch.from_numpy(np.ascontiguousarray(w)))



# Training subsets are drawn only from the NB01 fold-specific manifest.
import hashlib

# CONTENT signatures, not just paths/mtime: modified shapefile geometry or DBF
# triggers a new M2 training key and *both* M1/M2 evaluation keys.
def shapefile_content_signature(source):
    p=Path(source)
    if not p.is_file():raise FileNotFoundError(p)
    files=sorted(p.parent.glob(p.stem+'.*')) if p.suffix.lower()=='.shp' else [p]
    files=([x for x in files if x.suffix.lower() in ('.shp','.shx','.dbf','.prj','.cpg')]
           if p.suffix.lower()=='.shp' else [p])
    h=hashlib.sha256()
    for file in files:
        h.update(file.suffix.lower().encode())
        with file.open('rb') as stream:
            for chunk in iter(lambda:stream.read(4*1024*1024),b''):
                h.update(chunk)
    return h.hexdigest()

EXHAUSTIVE_SOURCE=Path(A['exhaustive_annotations']['positive_polygons'])
EXHAUSTIVE_SHA=shapefile_content_signature(EXHAUSTIVE_SOURCE)
print('Exhaustive annotation SHA256:',EXHAUSTIVE_SHA[:20],
      '| polygons loaded:',len(positive),flush=True)

CACHE_DIR=BASELINE/'chip_cache'
CACHE_DIR.mkdir(parents=True,exist_ok=True)
def cv_fingerprint(fold,model_type):
    sources=[REF/'01_cv_window_manifest.csv',REF/'01_circle_spatial_folds.csv',
             REF/'01_candidate_windows.csv',ROOT/'Outputs/V3/00_input_audit/00_input_manifest.json']
    for name in sorted(mosaics):
        sources.append(Path(mosaics[name]['path']))
    deps=[(str(p),p.stat().st_size,p.stat().st_mtime_ns) for p in sources]
    payload=dict(version='spatial3fold_v2',fold=int(fold),model=model_type,
                 holdout=sorted(HOLDOUT),dev=sorted(DEV),crop=CROP,seed=SEED,
                 max_partial=MAX_PARTIAL_PER_MOSAIC,
                 max_exhaustive=MAX_EXHAUSTIVE_PER_SITE,buffer_m=15,files=deps,
                 exhaustive_annotations_sha256=EXHAUSTIVE_SHA if model_type=='M2' else None)
    return hashlib.sha256(json.dumps(payload,sort_keys=True).encode()).hexdigest()

def make_cv_cache(model_type,fold):
    key=cv_fingerprint(fold,model_type)
    cache_file=CACHE_DIR/f'{model_type}_fold{fold}_{key[:20]}.npz'
    if cache_file.exists():
        try:
            old=_unpack_chips(cache_file,key)
            assert len(old)>15
            assert all(x['kind']=='partial' for x in old) if model_type=='M1' else True
            print('CACHE HIT',model_type,'fold',fold,len(old),'chips',flush=True)
            return old
        except Exception as e:
            print('Invalid full cache, rebuilding:',repr(e),flush=True)
            cache_file.unlink(missing_ok=True)
    # Same NB01 train candidate windows for both models, with exhaustive windows
    # added exclusively in M2; validation-fold windows are never eligible.
    eligible=splits[(splits.cv_fold==fold)&(splits.split=='train')].copy()
    eligible=eligible[eligible.kind.isin(['partial'] if model_type=='M1' else ['partial','exhaustive'])]
    if model_type=='M2':
        eligible=eligible[(eligible.kind=='partial')|eligible.circle_id.isin(DEV)]
    chosen=[]
    for (name,kind),group in eligible.groupby(['mosaic','kind']):
        cap=MAX_PARTIAL_PER_MOSAIC*(3 if geo[name]['site']=='MOS' else 1) if kind=='partial' else MAX_EXHAUSTIVE_PER_SITE
        group=group.sample(n=min(cap,len(group)),random_state=SEED)
        chosen.append(group)
    assert chosen,'No CV training windows'
    selected=pd.concat(chosen,ignore_index=True)
    assert not selected[selected.kind=='exhaustive'].circle_id.isin(HOLDOUT).any()
    print(model_type,'fold',fold,'selected windows:',selected.groupby(['site','kind']).size().to_dict(),flush=True)
    parts_dir=CACHE_DIR/f'{model_type}_fold{fold}_{key[:20]}'
    parts_dir.mkdir(parents=True,exist_ok=True)
    all_chips=[]
    for name,grp in selected.groupby('mosaic',sort=True):
        records=list(grp.itertuples(index=False))
        with staged_mosaic(name):
            for off in range(0,len(records),100):
                part=parts_dir/f'{name}_{off:04d}.npz'
                if part.exists():
                    try:
                        chunks=_unpack_chips(part,key)
                        all_chips.extend(chunks)
                        continue
                    except Exception:
                        part.unlink(missing_ok=True)
                items=[x for row in records[off:off+100] if (x:=training_crop(row,model_type)) is not None]
                if items:_pack_chips(part,items,key)
                all_chips.extend(items)
                print(model_type,'fold',fold,name,min(off+100,len(records)),'/',len(records),'usable',len(items),flush=True)
    assert len(all_chips)>15,(model_type,fold,len(all_chips))
    assert all(chip['kind']=='partial' for chip in all_chips) if model_type=='M1' else True
    _pack_chips(cache_file,all_chips,key)
    print('CACHE SAVED',model_type,'fold',fold,len(all_chips),'chips',flush=True)
    return all_chips


## 05 — Identical training recipe for M1 and M2; three spatial folds


In [ ]:
# V2 notebook 04 Section 04: stochastic photometric training augmentation.
# Dataset is applied ONLY during training. Inference remains native RGB, unchanged.
def v2_soft_shadow(x,rng):
    h,w=x.shape[:2]
    yy,xx=np.mgrid[:h,:w].astype('float32')
    theta=rng.uniform(0,np.pi)
    normal=np.cos(theta)*(xx-w/2)+np.sin(theta)*(yy-h/2)
    offset=rng.uniform(-.35,.35)*max(h,w)
    edge_width=rng.uniform(20,90)
    transition=np.clip((normal-offset)/edge_width,-12,12)
    weights=1/(1+np.exp(-transition))
    darkness=rng.uniform(.45,.90)
    return np.clip(x*(darkness+(1-darkness)*weights[:,:,None]),0,1)

class V2PhotometricDataset(Dataset):
    def __init__(self,rows): self.rows=rows
    def __len__(self):return len(self.rows)
    def __getitem__(self,i):
        q=self.rows[i];x=q['image'].copy();y=q['target'].copy();w=q['weight'].copy()
        # Separate reproducible stream per sample retrieval; same V2 parameter ranges.
        rng=np.random.default_rng(SEED + int(torch.randint(0,2**30,(1,)).item()) + i*101)
        k=int(rng.integers(0,4))
        x=np.rot90(x,k);y=np.rot90(y,k);w=np.rot90(w,k)
        if rng.random()<.5:x=x[:,::-1];y=y[:,::-1];w=w[:,::-1]
        if rng.random()<.5:x=x[::-1];y=y[::-1];w=w[::-1]
        x=np.ascontiguousarray(x.astype('float32'))
        x=np.clip(x*rng.uniform(.70,1.30),0,1)
        x=np.clip((x-.5)*rng.uniform(.80,1.20)+.5,0,1)
        x=np.power(x,float(rng.uniform(.75,1.35)))
        x=np.clip(x*rng.uniform(.92,1.08,size=(1,1,3)),0,1)
        if rng.random()<.55:x=v2_soft_shadow(x,rng)
        return (torch.from_numpy(np.ascontiguousarray(x.transpose(2,0,1)).astype('float32')),
                torch.from_numpy(np.ascontiguousarray(y).astype('float32')),
                torch.from_numpy(np.ascontiguousarray(w).astype('float32')))

import time
from contextlib import nullcontext
USE_AMP=(device=='cuda')
# float16 is fast on both T4 and L4; BCE is evaluated in fp32 for stability.
scaler=torch.amp.GradScaler('cuda',enabled=USE_AMP)
print('CUDA mixed precision:',USE_AMP,'| GPU:',torch.cuda.get_device_name(0),flush=True)


def fit_cv(model_type,fold):
    fold_out=BASE_OUT/f'fold_{fold}'
    fold_out.mkdir(parents=True,exist_ok=True)
    key=cv_fingerprint(fold,model_type)
    variant_key=hashlib.sha256((key+'NB02D2_V2_photo_v1').encode()).hexdigest()
    # Versioned M2 models cannot silently reuse old Yangambi labels.
    # M1 uses partial labels only. Reuse the original completed M1 checkpoint
    # under the explicit assumption that only exhaustive POSITIVE polygons changed.
    legacy_m1=fold_out/f'fold_{fold}'/'M1_resnet18_spatialCV.pt'
    ckpt=(legacy_m1 if model_type=='M1' and legacy_m1.exists() else
          fold_out/f'M1_resnet18_spatialCV.pt' if model_type=='M1' else
          fold_out/f'M2_v2_photometric_{variant_key[:20]}.pt')
    torch.manual_seed(SEED)
    model=ResNet18UNet().to(device)
    if ckpt.exists():
        model.load_state_dict(torch.load(ckpt,map_location=device,weights_only=True))
        print('Reusing spatial-CV checkpoint:',ckpt,flush=True)
        if model_type=='M1':
            print('M1 reuse assumes partial tree annotations, circle boundaries, '
                  'fold allocation and ROI unchanged.',flush=True)
        return model.eval()
    cache_file=BASELINE/'chip_cache'/f'{model_type}_fold{fold}_{key[:20]}.npz'
    assert cache_file.exists(),('Run NB02B first: original RGB chip cache missing',cache_file)
    chips=_unpack_chips(cache_file,key)
    loader=DataLoader(V2PhotometricDataset(chips),batch_size=BATCH,shuffle=True,num_workers=0,pin_memory=True)
    optim=torch.optim.AdamW(model.parameters(),lr=2e-4,weight_decay=1e-4)
    local_scaler=torch.amp.GradScaler('cuda',enabled=USE_AMP)
    history=[]
    for ep in range(EPOCHS):
        model.train();losses=[];started=time.perf_counter()
        for x,y,w in loader:
            x=x.to(device,non_blocking=True);y=y.to(device,non_blocking=True);w=w.to(device,non_blocking=True)
            optim.zero_grad(set_to_none=True)
            with torch.autocast(device_type='cuda',dtype=torch.float16,enabled=USE_AMP):
                logits=model(x)
            loss=masked_loss(logits.float(),y,w)
            local_scaler.scale(loss).backward()
            local_scaler.step(optim);local_scaler.update()
            losses.append(float(loss.detach()))
        avg=float(np.mean(losses))
        history.append(dict(epoch=ep+1,masked_BCE=avg,seconds=time.perf_counter()-started))
        pd.DataFrame(history).to_csv(fold_out/f'{model_type}_v2_photo_loss.csv',index=False)
        print(f'{model_type} fold {fold} epoch {ep+1:02d}/{EPOCHS} BCE={avg:.4f} {history[-1]["seconds"]:.1f}s',flush=True)
    torch.save(model.state_dict(),ckpt)
    del chips
    print('Saved:',ckpt,flush=True)
    return model.eval()


## 06 — Independent circle inference: each circle evaluated exactly once per model


In [ ]:
from scipy.special import expit,logit
from scipy.optimize import minimize
from sklearn.metrics import r2_score,mean_squared_error,mean_absolute_error
WINDOWS=pd.read_csv(REF/'01_candidate_windows.csv')
VAL_WINDOWS=WINDOWS[WINDOWS.kind=='exhaustive'].copy()
VAL_WINDOWS=VAL_WINDOWS[VAL_WINDOWS.circle_id.isin(links.circle_id)].copy()
assert set(VAL_WINDOWS.circle_id.unique().astype(int))==set(links.circle_id.astype(int))
N_BINS=256
BINS=np.linspace(0,1,N_BINS+1)
BIN_CENTER=(BINS[:-1]+BINS[1:])/2

def run_all_circles(model,model_name):
    statfile=OUT/f'{model_name}_spatialCV_circle_raw_{EVAL_KEY[:20]}.csv'
    histfile=OUT/f'{model_name}_spatialCV_histograms_{EVAL_KEY[:20]}.npz'
    if statfile.exists() and histfile.exists():
        return pd.read_csv(statfile),dict(np.load(histfile))
    model.eval();rows=[];histograms={}
    for _,circle in split_df.iterrows():
        cid=int(circle.circle_id)
        name=circle.mosaic
        d=geo[name]
        windows=VAL_WINDOWS[(VAL_WINDOWS.circle_id==cid)&(VAL_WINDOWS.mosaic==name)]
        assert len(windows),f'No evaluation windows for circle {cid}'
        cg=circles[circles.original_row==cid].to_crs(d['crs'])
        counts=np.zeros(N_BINS,np.int64);true_counts=np.zeros(N_BINS,np.int64)
        n=0;sumy=0.;sump=0.;sum_sq=0.;tp=fp=fn=tn=0
        rng=np.random.default_rng(SEED+cid)
        sample_truth=[];sample_probs=[]
        for rec in windows.itertuples(index=False):
            win=Window(int(rec.col),int(rec.row),int(rec.width),int(rec.height))
            mask=burn_window(cg.geometry,name,win)
            mask &= burn_window(d['roi_local'].geometry,name,win)
            rgb,valid=rgb_for_window(name,win)
            mask &= valid
            if not mask.any():continue
            reference=burn_window(d['positive_local'].geometry,name,win)
            x=torch.from_numpy(rgb.transpose(2,0,1).copy())[None].to(device)
            with torch.inference_mode():
                pred=torch.sigmoid(model(x))[0].cpu().numpy()
            y=reference[mask].astype('uint8');p=pred[mask].astype('float32')
            n+=len(y);sumy+=float(y.sum());sump+=float(p.sum())
            sum_sq+=float(np.square(p-y).sum())
            binary=p>=.5
            tp+=int(np.sum(binary&(y==1)));fp+=int(np.sum(binary&(y==0)))
            fn+=int(np.sum((~binary)&(y==1)))
            tn+=int(np.sum((~binary)&(y==0)))
            ix=np.minimum((p*N_BINS).astype(np.int32),N_BINS-1)
            counts+=np.bincount(ix,minlength=N_BINS).astype('int64')
            true_counts+=np.bincount(ix,weights=y,minlength=N_BINS).astype('int64')
            # only diagnostic AUC/AP uses pixel subsampling, with fixed per-circle cap
            take=rng.choice(len(y),min(100,len(y)),replace=False)
            sample_truth.extend(y[take].tolist());sample_probs.extend(p[take].tolist())
        assert n>0,f'No valid ROI/RGB reference pixels for circle {cid}'
        assert int(counts.sum())==n
        yt=np.asarray(sample_truth);pr=np.asarray(sample_probs)
        rows.append(dict(model=model_name,circle_id=cid,site=circle.site,split=circle.split,
                         valid_pixels=n,observed_fraction=100*sumy/n,
                         predicted_fraction=100*sump/n,
                         bias_pp=100*(sump-sumy)/n,brier=sum_sq/n,
                         f1_threshold_05=2*tp/max(1,2*tp+fp+fn),
                         tp=tp,fp=fp,fn=fn,tn=tn,
                         iou_threshold_05=tp/max(1,tp+fp+fn),
                         precision_threshold_05=tp/max(1,tp+fp),
                         recall_threshold_05=tp/max(1,tp+fn),
                         auc_sample=roc_auc_score(yt,pr) if len(np.unique(yt))>1 else np.nan,
                         ap_sample=average_precision_score(yt,pr) if len(np.unique(yt))>1 else np.nan))
        histograms[str(cid)+'_count']=counts
        histograms[str(cid)+'_positive']=true_counts
        print(model_name,'circle',cid,'obs',round(100*sumy/n,1),
              'pred',round(100*sump/n,1),'valid pixels',n,flush=True)
    frame=pd.DataFrame(rows)
    assert frame.circle_id.is_unique
    assert set(frame.circle_id.astype(int))==set(split_df.circle_id.astype(int))
    assert len(frame)==len(split_df)
    frame.to_csv(statfile,index=False)
    np.savez_compressed(histfile,**histograms)
    return frame,histograms



## 07 — Frozen baseline comparison: V2 photometric training, native RGB evaluation


In [ ]:
import hashlib,gc
from sklearn.metrics import r2_score,mean_absolute_error,mean_squared_error
SITES=['Baego','MOS','Yangambi','Yoko']
FOLDS=[0,1,2]
BASE_OUT=OUT
version_file=BASELINE/'02B_latest_reference_version.json'
assert version_file.exists()
assert json.loads(version_file.read_text())['sha256']==EXHAUSTIVE_SHA
old=pd.read_csv(BASELINE/'02B_spatialCV_all30_paired.csv')
old=old[old.model=='M2'].copy()
assert len(old)==30 and old.circle_id.is_unique
ALL_SCORES=[]
for fold in FOLDS:
    FOLD=int(fold)
    HOLDOUT=set(links.loc[links.fold==FOLD,'circle_id'].astype(int))
    DEV=set(links.loc[links.fold!=FOLD,'circle_id'].astype(int))
    split_df=links[links.fold==FOLD][['circle_id','site','mosaic','cover_pct']].copy()
    split_df['split']='held_out_spatial_fold'
    for name,d in geo.items():
        cs=circles.to_crs(d['crs'])
        d['dev_circles']=cs[cs.original_row.isin(DEV)]
        BLOCKED[name]=[g.buffer(15) for g in cs[cs.original_row.isin(HOLDOUT)].geometry]
    OUT=BASE_OUT/f'fold_{FOLD}'
    OUT.mkdir(parents=True,exist_ok=True)
    model_type='M2'
    EVAL_KEY=hashlib.sha256((
        cv_fingerprint(FOLD,model_type)+'NB02D2_photo_v1_eval').encode()).hexdigest()
    print('\nPHOTO FOLD',FOLD,'training or resuming',flush=True)
    model=fit_cv(model_type,FOLD)
    result,_=run_all_circles(model,'M2_v2_photo')
    result['fold']=FOLD
    assert set(result.circle_id.astype(int))==HOLDOUT
    ALL_SCORES.append(result)
    pd.concat(ALL_SCORES,ignore_index=True).to_csv(
        BASE_OUT/'02D2_progress.csv',index=False)
    del model;torch.cuda.empty_cache();gc.collect()

NEW=pd.concat(ALL_SCORES,ignore_index=True)
NEW.to_csv(BASE_OUT/'02D2_v2_photometric_30circles.csv',index=False)
raw=old.set_index('circle_id').sort_index()
pho=NEW.set_index('circle_id').sort_index()
assert raw.index.equals(pho.index)
assert (raw.site==pho.site).all() and (raw.fold==pho.fold).all()
assert np.array_equal(raw.valid_pixels.to_numpy(),pho.valid_pixels.to_numpy())
assert np.allclose(raw.observed_fraction,pho.observed_fraction,atol=1e-5)
paired=pd.DataFrame(dict(circle_id=raw.index,site=raw.site.values,
    fold=raw.fold.values,observed_fraction=raw.observed_fraction.values,
    native_m2_fraction=raw.predicted_fraction.values,
    photo_m2_fraction=pho.predicted_fraction.values))
paired['native_abs_error_pp']=(paired.native_m2_fraction-paired.observed_fraction).abs()
paired['photo_abs_error_pp']=(paired.photo_m2_fraction-paired.observed_fraction).abs()
paired['gain_pp']=paired.native_abs_error_pp-paired.photo_abs_error_pp
paired.to_csv(BASE_OUT/'02D2_paired_30circles.csv',index=False)

def summary_one(g,variant,site):
    y=g.observed_fraction.to_numpy();p=g.predicted_fraction.to_numpy()
    tp,fp,fn=[float(g[k].sum()) for k in ('tp','fp','fn')]
    return dict(variant=variant,site=site,n=len(g),
        R2=r2_score(y,p) if len(g)>1 else np.nan,
        MAE_pp=mean_absolute_error(y,p),
        RMSE_pp=np.sqrt(mean_squared_error(y,p)),
        bias_pp=float((p-y).mean()),
        pixel_F1=2*tp/max(1,2*tp+fp+fn),
        pixel_IoU=tp/max(1,tp+fp+fn))
res=[]
for variant,frame in [('M2_native_NB02B',old),('M2_V2_photo_NB02D2',NEW)]:
    for site in ['ALL']+SITES:
        sub=frame if site=='ALL' else frame[frame.site==site]
        res.append(summary_one(sub,variant,site))
METRICS=pd.DataFrame(res)
METRICS.to_csv(BASE_OUT/'02D2_metrics_by_site.csv',index=False)
display(METRICS.round(4))
print('Photo augmentation better absolute error in',
      int((paired.gain_pp>0).sum()),'of 30 held-out circles')


## 08 — Final V2-photometric M2 trained on all 30 circles (production checkpoint)

**Run after NB02B's final all-30 native-RGB cache has been created.** This section consumes the **same 1,760 native RGB 256×256 chips** already cached in NB02B Section 08, but trains a *new* ImageNet-initialized M2 using the **V2 brightness/contrast/gamma/channel-gain and soft-shadow augmentations implemented above**, rather than local illumination correction. No fold CV is rerun. The all-circle training data are not a held-out validation dataset; report NB02D2's preceding spatial-CV results as its performance estimate. Do not run the full Section 07 CV training loop again.

Training: 25 epochs, same AdamW 2e−4, weight decay 1e−4, batch size 8 and masked BCE as NB02B, with reproducible seeded augmentation. Checkpoint and loss history live in `Outputs/V3/02D2_v2_photometric_M2/final_all30_M2/`. The final native-RGB M2 remains unchanged. **NB03 can explicitly select this photometric checkpoint in its setup via `MODEL_MODE='photo_all30'`**. Fold checkpoints, previous scores and native-RGB outputs stay independent.


In [ ]:
# Final all-30 *PHOTOMETRIC* production model, reusing frozen NB02B 30-circle RGB chips.
# Run 02D2 setup/definitions through Section 05, skip Section 07's CV loop.
import hashlib, json, time, gc
from torch.utils.data import DataLoader

assert CROP==256 and EPOCHS==25 and BATCH==8 and SEED==20261016
native_root=BASELINE/'final_all30_M2'
native_manifest_path=native_root/'02B_final_all30_manifest.json'
assert native_manifest_path.exists(),(
    'First complete NB02B Section 08 (final all-30 native RGB)',native_manifest_path)
nm=json.loads(native_manifest_path.read_text())
assert nm['exhaustive_annotations_sha256']==EXHAUSTIVE_SHA,(
    'NB02B final chips and NB02D2 exhaustive annotation bytes differ')
assert len(nm['circle_ids'])==30 and set(map(int,nm['circle_ids']))==set(links.circle_id.astype(int))
assert nm['crop']==CROP and nm['epochs']==EPOCHS and nm['batch']==BATCH
native_cache=Path(nm['chip_cache'])
assert native_cache.exists(),native_cache
native_key=nm['training_key']
print('Reading frozen NB02B all-30 native RGB chips:',native_cache,flush=True)
chips=_unpack_chips(native_cache,native_key)
assert len(chips)>30
assert set(q['site'] for q in chips)=={'Baego','MOS','Yangambi','Yoko'}
assert all(q['image'].shape==(CROP,CROP,3) for q in chips)

FINAL_PHOTO_ROOT=BASELINE.parent/'02D2_v2_photometric_M2'/'final_all30_M2'
FINAL_PHOTO_ROOT.mkdir(parents=True,exist_ok=True)
photo_config=dict(experiment='02D2_final_all30_V2photometric_v1',
    baseline_all30_key=native_key,baseline_training_manifest=nm,
    annotations_sha256=EXHAUSTIVE_SHA,architecture='ResNet18UNet',
    augmentation='V2_04_brightness_contrast_gamma_channelgain_softshadow',
    inference_input='native_RGB_0_to_1_no_ImageNet_mean_std',
    source_chips=str(native_cache),n_chips=len(chips),
    crop=CROP,epochs=EPOCHS,batch=BATCH,seed=SEED,
    lr=2e-4,weight_decay=1e-4)
photo_key=hashlib.sha256(json.dumps(photo_config,sort_keys=True).encode()).hexdigest()
FINAL_PHOTO_MODEL=FINAL_PHOTO_ROOT/f'M2_FINAL_V2photo_all30_{photo_key[:20]}.pt'
final_meta=FINAL_PHOTO_ROOT/'02D2_final_all30_manifest.json'
if final_meta.exists():
    old=json.loads(final_meta.read_text())
    assert old['training_key']==photo_key,(
        'Existing photometric final checkpoint uses a different training signature',final_meta)
else:
    final_meta.write_text(json.dumps(dict(photo_config,training_key=photo_key,
                               checkpoint=str(FINAL_PHOTO_MODEL)),indent=2))
if FINAL_PHOTO_MODEL.exists():
    print('Final V2-photometric checkpoint already exists:',FINAL_PHOTO_MODEL,flush=True)
else:
    torch.manual_seed(SEED)
    np.random.seed(SEED)
    model=ResNet18UNet().to(device)
    loader=DataLoader(V2PhotometricDataset(chips),batch_size=BATCH,
                      shuffle=True,num_workers=0,pin_memory=True)
    optimizer=torch.optim.AdamW(model.parameters(),lr=2e-4,weight_decay=1e-4)
    scaler=torch.amp.GradScaler('cuda',enabled=device=='cuda')
    history=[]
    for epoch in range(EPOCHS):
        model.train();losses=[];start=time.perf_counter()
        for x,y,w in loader:
            x=x.to(device,non_blocking=True)
            y=y.to(device,non_blocking=True)
            w=w.to(device,non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type='cuda',dtype=torch.float16,enabled=device=='cuda'):
                logits=model(x)
            loss=masked_loss(logits.float(),y,w)
            scaler.scale(loss).backward()
            scaler.step(optimizer);scaler.update()
            losses.append(float(loss.detach()))
        epoch_loss=float(np.mean(losses))
        assert np.isfinite(epoch_loss),('Non-finite training loss',epoch)
        history.append(dict(epoch=epoch+1,masked_BCE=epoch_loss,
                            seconds=time.perf_counter()-start))
        pd.DataFrame(history).to_csv(
            FINAL_PHOTO_ROOT/'02D2_final_all30_training_history.csv',index=False)
        print(f'FINAL V2 PHOTO epoch {epoch+1:02d}/{EPOCHS} '
              f'BCE={epoch_loss:.4f} {history[-1]["seconds"]:.1f}s',flush=True)
    temp=FINAL_PHOTO_MODEL.with_suffix('.partial.pt')
    torch.save(model.state_dict(),temp)
    temp.replace(FINAL_PHOTO_MODEL)
    del model
    gc.collect()
    torch.cuda.empty_cache()
    print('Saved final V2 photometric M2:',FINAL_PHOTO_MODEL,flush=True)
del chips
gc.collect()
print('PHOTO FINAL READY:',FINAL_PHOTO_MODEL)
print('NB03 selection: MODEL_MODE="photo_all30"')
